In [1]:
# ==================================================
# PHASE 4
# EVIDENCE DOSSIER + AUDIT REPORT GENERATION
# ==================================================

import pandas as pd
import numpy as np
import json

# ==================================================
# LOAD DATA
# ==================================================

df = pd.read_csv("pseudo_labeled_tickets.csv")

print("Dataset Shape:", df.shape)

# ==================================================
# SEVERITY MAPPING
# ==================================================

severity_map = {
    "Low": 1,
    "Medium": 2,
    "High": 3,
    "Critical": 4
}

# ==================================================
# ENSURE NUMERIC SEVERITY COLUMNS EXIST
# ==================================================

if "priority_numeric" not in df.columns:
    df["priority_numeric"] = (
        df["Priority_Level"]
        .map(severity_map)
    )

if "inferred_numeric" not in df.columns:
    df["inferred_numeric"] = (
        df["inferred_severity"]
        .map(severity_map)
    )

# ==================================================
# SEVERITY DELTA
# ==================================================

df["severity_delta"] = (
    df["inferred_numeric"]
    -
    df["priority_numeric"]
)

# ==================================================
# MISMATCH TYPE
# ==================================================

def determine_mismatch_type(row):

    if row["severity_delta"] > 0:
        return "Hidden Crisis"

    elif row["severity_delta"] < 0:
        return "False Alarm"

    return "Consistent"

df["mismatch_type"] = (
    df.apply(
        determine_mismatch_type,
        axis=1
    )
)

# ==================================================
# EVIDENCE GENERATOR
# ==================================================

def generate_evidence(row):

    evidence = []

    # Resolution Time Evidence

    if row["Resolution_Time_Hours"] > 58:

        evidence.append(
            f"Resolution time extremely high ({row['Resolution_Time_Hours']} hrs)"
        )

    elif row["Resolution_Time_Hours"] > 27:

        evidence.append(
            f"Resolution time elevated ({row['Resolution_Time_Hours']} hrs)"
        )

    # Satisfaction Score Evidence

    if row["Satisfaction_Score"] <= 2:

        evidence.append(
            f"Very low customer satisfaction ({row['Satisfaction_Score']})"
        )

    elif row["Satisfaction_Score"] == 3:

        evidence.append(
            "Moderate customer dissatisfaction"
        )

    # Keyword Evidence

    if "keyword_score" in row.index:

        if row["keyword_score"] >= 4:

            evidence.append(
                "Multiple high severity keywords detected"
            )

        elif row["keyword_score"] > 0:

            evidence.append(
                "Severity-related keywords detected"
            )

    # Category Evidence

    if "category_signal" in row.index:

        if row["category_signal"] >= 3:

            evidence.append(
                f"Historically severe issue category ({row['Issue_Category']})"
            )

    # Embedding Evidence

    if "embedding_signal" in row.index:

        if row["embedding_signal"] >= 3:

            evidence.append(
                "Embedding model indicates high severity similarity"
            )

    if len(evidence) == 0:

        evidence.append(
            "No strong supporting evidence found"
        )

    return evidence

# ==================================================
# GENERATE EVIDENCE COLUMN
# ==================================================

df["evidence"] = (
    df.apply(
        generate_evidence,
        axis=1
    )
)

# ==================================================
# CREATE DOSSIER OBJECT
# ==================================================

def create_dossier(row):

    return {

        "ticket_id":
            str(row["Ticket_ID"]),

        "assigned_priority":
            str(row["Priority_Level"]),

        "inferred_severity":
            str(row["inferred_severity"]),

        "mismatch_type":
            str(row["mismatch_type"]),

        "severity_delta":
            int(row["severity_delta"]),

        "resolution_time_hours":
            float(row["Resolution_Time_Hours"]),

        "satisfaction_score":
            float(row["Satisfaction_Score"]),

        "issue_category":
            str(row["Issue_Category"]),

        "evidence":
            row["evidence"]

    }

# ==================================================
# GENERATE DOSSIERS
# ==================================================

dossiers = []

for _, row in df.iterrows():

    if row["mismatch"] == 1:

        dossiers.append(
            create_dossier(row)
        )

print(
    "\nTotal Evidence Dossiers:",
    len(dossiers)
)

# ==================================================
# SAVE DOSSIER JSON
# ==================================================

with open(
    "evidence_dossier.json",
    "w"
) as f:

    json.dump(
        dossiers,
        f,
        indent=4
    )

print(
    "Saved: evidence_dossier.json"
)

# ==================================================
# AUDIT REPORT
# ==================================================

audit_df = df[
    [

        "Ticket_ID",

        "Priority_Level",

        "inferred_severity",

        "mismatch",

        "mismatch_type",

        "severity_delta",

        "Resolution_Time_Hours",

        "Satisfaction_Score",

        "Issue_Category"

    ]
]

audit_df.to_csv(
    "audit_report.csv",
    index=False
)

print(
    "Saved: audit_report.csv"
)

# ==================================================
# SUMMARY STATISTICS
# ==================================================

summary = {

    "total_tickets":
        int(len(df)),

    "total_mismatches":
        int(
            df["mismatch"].sum()
        ),

    "consistent_tickets":
        int(
            (df["mismatch"] == 0).sum()
        ),

    "hidden_crisis":
        int(
            (
                df["mismatch_type"]
                ==
                "Hidden Crisis"
            ).sum()
        ),

    "false_alarm":
        int(
            (
                df["mismatch_type"]
                ==
                "False Alarm"
            ).sum()
        )
}

print("\n==============================")
print("AUDIT SUMMARY")
print("==============================")

for k, v in summary.items():
    print(f"{k}: {v}")

# ==================================================
# TOP HIDDEN CRISIS CASES
# ==================================================

hidden_crisis = df[
    df["mismatch_type"] ==
    "Hidden Crisis"
]

print("\nTop Hidden Crisis Cases")

print(

    hidden_crisis[
        [
            "Ticket_ID",
            "Priority_Level",
            "inferred_severity",
            "severity_delta"
        ]
    ]
    .head(10)

)

# ==================================================
# TOP FALSE ALARMS
# ==================================================

false_alarm = df[
    df["mismatch_type"] ==
    "False Alarm"
]

print("\nTop False Alarm Cases")

print(

    false_alarm[
        [
            "Ticket_ID",
            "Priority_Level",
            "inferred_severity",
            "severity_delta"
        ]
    ]
    .head(10)

)

print("\nPHASE 4 COMPLETED SUCCESSFULLY")

Dataset Shape: (20000, 32)

Total Evidence Dossiers: 12009
Saved: evidence_dossier.json
Saved: audit_report.csv

AUDIT SUMMARY
total_tickets: 20000
total_mismatches: 12009
consistent_tickets: 7991
hidden_crisis: 1489
false_alarm: 10520

Top Hidden Crisis Cases
     Ticket_ID Priority_Level inferred_severity  severity_delta
3   TKT-100003            Low            Medium               1
11  TKT-100011            Low            Medium               1
14  TKT-100014            Low            Medium               1
27  TKT-100027            Low            Medium               1
34  TKT-100034            Low            Medium               1
38  TKT-100038            Low            Medium               1
67  TKT-100067            Low            Medium               1
72  TKT-100072            Low            Medium               1
82  TKT-100082            Low            Medium               1
85  TKT-100085            Low            Medium               1

Top False Alarm Cases
     Ticket_

In [3]:
print(summary)
df["mismatch_type"].value_counts()

{'total_tickets': 20000, 'total_mismatches': 12009, 'consistent_tickets': 7991, 'hidden_crisis': 1489, 'false_alarm': 10520}


,count
mismatch_type,
False Alarm,10520
Consistent,7991
Hidden Crisis,1489
